In [18]:
import surprise
import numpy as np
import pandas as pd

In [4]:
ratings = pd.read_csv(r"F:\MACHINE LEARNING\Cases_rec_sys\filmtrust\ratings.txt", sep = ' ', names = ['uid','iid','rating'])
ratings.head()

,uid,iid,rating
0,1,1,2.0
1,1,2,4.0
2,1,3,3.5
3,1,4,3.0
4,1,5,4.0


In [6]:
lowest_rating = ratings['rating'].min()
highest_rating=ratings['rating'].max()
lowest_rating,highest_rating

(0.5, 4.0)

In [7]:
reader = surprise.Reader(rating_scale = (lowest_rating, highest_rating))
data = surprise.Dataset.load_from_df(ratings, reader)
type(data)

surprise.dataset.DatasetAutoFolds

In [8]:
similarity_options = {'name': 'cosine', 'user_based': True}
algo = surprise.KNNBasic(sim_options = similarity_options) # default k = 40
output = algo.fit(data.build_full_trainset())
# calculates the expected ratings for all the users

Computing the cosine similarity matrix...
Done computing similarity matrix.


In [10]:
from surprise.model_selection import train_test_split

In [12]:
trainset, testset = train_test_split(data, test_size= 0.25, random_state=26)
similarity_options = {'name': 'cosine', 'user_based': True}
algo = surprise.KNNBasic(sim_options = similarity_options, k=70) # default k = 40
algo.fit(trainset)
predictions = algo.test(testset)
surprise.accuracy.rmse(predictions)

Computing the cosine similarity matrix...
Done computing similarity matrix.
RMSE: 0.9282


0.9282056050129103

### Tuning for best K

In [9]:
from surprise.model_selection import GridSearchCV
from surprise.model_selection.split import KFold

In [25]:
param_grid = {'k': [25,40,50,65,75,85], 'user_bases': [False, True]}
kfold = KFold(n_splits = 5, random_state = 26, shuffle = True)
gs = GridSearchCV(surprise.KNNWithZScore, param_grid , measures = ['rmse','mae'], cv = kfold)
gs.fit(data)

Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computi

In [26]:
print(gs.best_score['rmse'])
print(gs.best_params['rmse'])

0.82858464566139
{'k': 85, 'user_bases': False}


#### Building best model on whole data

In [16]:
algo = gs.best_estimator['rmse']
algo.fit(data.build_full_trainset())

Computing the msd similarity matrix...
Done computing similarity matrix.


Total number of items

In [19]:
a = [2,4,5,7,8,9]
b = [4,7,9]
np.setdiff1d(a,b)

array([2, 5, 8])

In [21]:
iids = ratings['iid'].unique()
print(len(iids))

2071


In [22]:
user = 50
u_iid = ratings[ratings['uid']==user]['iid'].unique()
print('List of items rated by user: \n', u_iid)
print('No. of items rated by user {0}: {1}'.format(user,  len(u_iid)))
iids_to_predict = np.setdiff1d(iids, u_iid)
print("Items not rated by the user or those items for which the expexted ratings are to be predicted: ", iids_to_predict)

List of items rated by user: 
 [  8 211   3   2 219 234  12 254 250 207  11 253 236  84  10   7 233  13
   1   5   6 252 241 216 257 206   4 217   9 215 213  17 255 220 121 245
 239 251 235]
No. of items rated by user 50: 39
Items not rated by the user or those items for which the expexted ratings are to be predicted:  [  14   15   16 ... 2069 2070 2071]


In [24]:
testset = [[user, iid, 0.] for iid in iids_to_predict]
predictions = algo.test(testset)
exp_ratings = [(predictions[i].iid, predictions[i].est) for i in range(0, len(predictions))]
exp_ratings = pd.DataFrame(exp_ratings,columns=['iid','est_rating'])
exp_ratings.sort_values(by='est_rating',ascending=False).head(30)

,iid,est_rating
1282,1322,4.0
1655,1695,4.0
1413,1453,4.0
1409,1449,4.0
1911,1951,4.0
1910,1950,4.0
1826,1866,4.0
1404,1444,4.0
1829,1869,4.0
1403,1443,4.0
